# Experimentos de Modelos - MLflow
Este notebook se apoya en el paquete `src/` para ejecutar los diversos algoritmos predictivos incorporando balanceo de clases (SMOTE, OverSampling, class_weight) con GridSearchCV.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.data_pipeline import load_and_preprocess_data
from src.model_pipeline import evaluar_y_registrar
from src.config import get_model_configs

### Carga de datos con Pipeline Data

In [2]:
df, encoders = load_and_preprocess_data('../WA_Fn-UseC_-Telco-Customer-Churn.csv')
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

ratio_neg_pos = sum(y_train==0)/sum(y_train==1)

### GridSearch con config exportada en `src/config.py`

In [3]:
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold

cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
modelos_cfg = get_model_configs(ratio_neg_pos=ratio_neg_pos)

resultados = []
for nombre_modelo, cfg in modelos_cfg.items():
    print(f"-> Optimizando y evaluando: {nombre_modelo}")
    grid = GridSearchCV(
        estimator=cfg['estimator'],
        param_grid=cfg['param_grid'],
        cv=cv_strategy,
        scoring='f1',
        n_jobs=-1, verbose=1, refit=True
    )
    grid.fit(X_train_scaled, y_train)
    
    mejores_params = grid.best_params_
    print(f"Top Parametros para {nombre_modelo}: {mejores_params}")
    
    res = evaluar_y_registrar(grid.best_estimator_, nombre_modelo, X_train_scaled, X_test_scaled, y_train, y_test, best_params=mejores_params)
    resultados.append(res)

df_res = pd.DataFrame(resultados).sort_values(by='f1', ascending=False)
df_res

-> Optimizando y evaluando: DecisionTree
Fitting 10 folds for each of 18 candidates, totalling 180 fits
Top Parametros para DecisionTree: {'classifier__class_weight': None, 'classifier__max_depth': 5, 'resampler': RandomOverSampler(random_state=42)}


2026/03/28 12:12:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 12:12:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


-> Optimizando y evaluando: RandomForest
Fitting 10 folds for each of 16 candidates, totalling 160 fits
Top Parametros para RandomForest: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 5, 'classifier__n_estimators': 50, 'resampler': None}


2026/03/28 12:13:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 12:13:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


-> Optimizando y evaluando: XGBoost
Fitting 10 folds for each of 12 candidates, totalling 120 fits


...

-> Optimizando y evaluando: SVM
Fitting 10 folds for each of 8 candidates, totalling 80 fits
Top Parametros para SVM: {'classifier__C': 0.1, 'classifier__class_weight': None, 'resampler': SMOTE(random_state=42)}


2026/03/28 12:19:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 12:19:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


-> Optimizando y evaluando: NaiveBayes
Fitting 10 folds for each of 2 candidates, totalling 20 fits
Top Parametros para NaiveBayes: {'resampler': None}


2026/03/28 12:20:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 12:20:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,model_name,precision,accuracy,recall,f1,auc
1,RandomForest,0.492659,0.727790,0.807487,0.611955,0.832112
2,XGBoost,0.497470,0.732054,0.788770,0.610134,0.827878
0,DecisionTree,0.510909,0.742715,0.751337,0.608225,0.820507
4,SVM,0.506239,0.739161,0.759358,0.607487,0.818566
5,NaiveBayes,0.498155,0.732765,0.721925,0.589520,0.811997
3,KNN,0.451104,0.690121,0.764706,0.567460,0.781841
